<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">从零开始构建推理模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第五章：习题解答

本笔记本中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.22.2


&nbsp;
## 练习 5.1：在自一致性中使用启发式评分器作为平局决胜

- 实现这一功能有多种方式
- 或许最简单的方法是在自一致性函数之外进行处理，并使用返回的字典（例如，类似于我们在练习4.4中实现打破平局时所做的那样，当时我们直接将其添加到了 `evaluate_math500_stream` 函数中）
- 相关代码行如下所示

```python
# ...
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with heuristic_score
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            # ...

    # ...
    return num_correct, num_examples, acc
```

- 第3章中相对于基线的改进以及第4章中的一致性结果如下所示

|   | 方法                                   | 模型 | 准确率 | 耗时      |
|---|------------------------------------------|-------|----------|-----------|
| 1 | 第4章基线（使用思维链提示）    | 基础模型  | 33.4%    | 129.2 分钟 |
| 2 | 一致性（n=3）+ 多数投票   | 基础模型  | 43.2%    | 328.2 分钟 |
| 3 | 一致性（n=3）+ 启发式方法       | 基础模型  | 43.4%    | 326.5 分钟 |
| 4 | 一致性（n=3）+ 平均对数概率    | 基础模型  | 44.8%    | 327.7 分钟 |

- 表中显示的准确率值和运行时间是在MATH-500测试集的全部500个样本上计算得出的，使用的是"cuda" GPU（DGX Spark）

- 为方便起见，您可以运行位于[../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts)目录下的[self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_consistency_scorer_math500.py)脚本

- 然而请注意，正如[#159](https://github.com/rasbt/reasoning-from-scratch/issues/159)中所讨论的，我们基于启发式分数决定多数获胜者，但仅考虑每个多数对中的第一个实例
- 例如

&nbsp;
## 练习 5.2：在最佳N选设置中使用启发式评分器

- Best-of-N 方法与自洽性类似，都会生成多个答案
- 然而，它并非基于多数投票选择最终答案，而是通过评分函数（如 `heuristic_score`）对所有答案进行评分，并返回得分最高的答案
- 实现有多种方式，但最简单的方法可能是以第四章现有的自洽性函数为模板，如以下所示替换为 `heuristic_score`

```python
# ...

from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- 结果如下所示

|   | 方法                                     | 模型 | 准确率 | 时间      |
|---|------------------------------------------|------|--------|-----------|
| 1 | 基于思维链提示的基线方法                   | Base | 33.4%  | 129.2 分钟 |
| 2 | Best-of-N (n=3) + 启发式方法               | Base | 40.6%  | 327.7 分钟 |
| 3 | Best-of-N (n=3) + 平均对数概率             | Base | 43.2%  | 330.2 分钟 |

- 表中展示的准确率值和运行时间是在 MATH-500 测试集的全部 500 个样本上，使用 "cuda" GPU（DGX Spark）计算得出的

- 为方便起见，您可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 目录下的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.3：在自一致性中使用对数概率评分器作为平局决胜机制

- 代码与练习 5.1 类似，只是我们将 `heuristic_score` 替换为 `avg_logprob_answer`

```python
# ...
# from reasoning_from_scratch.ch05 import heuristic_score
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            ### NEW: Break tie with avg_logprob_answer
            else:
                best = None
                best_score = float("-inf")
            
                # Consider all members of each majority group
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            # ...

    # ...
    return num_correct, num_examples, acc
```

- 第3章中对基线模型的改进以及第4章中的自一致性方法结果如下所示

|   | 方法                                     | 模型 | 准确率 | 运行时间   |
|---|------------------------------------------|------|--------|------------|
| 1 | 基线模型（思维链提示）                   | 基础 | 33.4%  | 129.2 分钟 |
| 2 | 自一致性 (n=3) + 多数投票                 | 基础 | 43.2%  | 328.2 分钟 |
| 3 | 自一致性 (n=3) + 启发式规则               | 基础 | 43.4%  | 326.5 分钟 |
| 4 | 自一致性 (n=3) + 平均对数概率             | 基础 | 44.8%  | 327.7 分钟 |

- 表中展示的准确率值和运行时间基于 MATH-500 测试集的全部 500 个样本计算，使用 "cuda" GPU（DGX Spark）

- 为方便操作，您可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 目录下的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.4：在 Best-of-N 设置中使用 logprob 评分器

- 要实现带有对数概率评分器的Best-of-N方法，我们可以使用练习5.2中的代码，并将`heuristic_score`替换为`avg_logprob_answer`：

```python

from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- 结果如下所示

| # | 方法                                     | 模型 | 准确率 | 用时      |
|---|------------------------------------------|------|--------|-----------|
| 1 | 基于思维链提示的基线方法                   | 基础 | 33.4%  | 129.2 分钟 |
| 2 | Best-of-N (n=3) + 启发式方法              | 基础 | 待定   | 待定      |
| 3 | Best-of-N (n=3) + 平均对数概率             | 基础 | 待定   | 待定      |

- 表中显示的准确率和运行时间是在 MATH-500 测试集的全部 500 个样本上计算得出的，使用的是 "cuda" GPU (DGX Spark)

- 为方便起见，您可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 目录下的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/best_of_n_math500.py) 脚本

&nbsp;
## 练习 5.5：使用启发式评分进行自我精炼

- 使用 `heuristic_score` 实际上比使用对数概率分数更简单，我们只需修改以下代码：

```python
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)


torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 更新后的代码为：

```python
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # NEW
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
```

- 使用启发式评分器的结果显示在第4、5和10行：

|    | 方法                   | 评分          | 迭代次数   | 模型       | 准确率   | 耗时      |
|----|------------------------|---------------|------------|------------|----------|-----------|
| 1  | 基线（第3章）          | -             | -          | 基础模型   | 15.2%    | 10.1 分钟 |
| 2  | 自我优化               | 无            | 1          | 基础模型   | 25.0%    | 84.8 分钟 |
| 3  | 自我优化               | 无            | 2          | 基础模型   | 22.0%    | 165.4 分钟|
| 4  | 自我优化               | 启发式        | 1          | 基础模型   | 21.6%    | 84.7 分钟 |
| 5  | 自我优化               | 启发式        | 2          | 基础模型   | 20.8%    | 151.4 分钟|
| 6  | 自我优化               | 平均对数概率  | 1          | 基础模型   | 21.4%    | 85.3 分钟 |
| 7  | 自我优化               | 平均对数概率  | 2          | 基础模型   | 22.0%    | 165.3 分钟|
|    |                        |               |            |            |          |           |
| 8  | 基线（第3章）          | -             | -          | 推理模型   | 48.2%    | 182.1 分钟|
| 9  | 自我优化               | 无            | 1          | 推理模型   | 56.6%    | 498.8 分钟|
| 10 | 自我优化               | 启发式        | 1          | 推理模型   | 57.8%    | 498.6 分钟|
| 11 | 自我优化               | 平均对数概率  | 1          | 推理模型   | 48.4%    | 499.7 分钟|

- 表中显示的准确率和运行时间是在 MATH-500 测试集的全部 500 个样本上计算得出的，使用的是 "cuda" GPU（DGX Spark）
- 为方便起见，您可以运行位于 [../02_math500-more-inference-scaling-scripts](../02_math500-more-inference-scaling-scripts) 目录下的 [self_consistency_scorer_math500.py](../02_math500-more-inference-scaling-scripts/self_refinement_math500.py) 脚本